# Aura Fizz asset generator (free)

1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Run **cell 1**. It installs, then **restarts by itself** (a "session crashed/restarted" message is normal).
3. Run **cell 2**, then **cell 3**, then **cell 4**.

The model is shrunk to fit free Colab's memory. Every image uses `coffee_front_final.png` as the style reference.

In [ ]:
# 1. Install, then auto-restart (about 2 minutes)
!pip -q install diffusers==0.41.0 "transformers>=4.56" accelerate bitsandbytes sentencepiece
print("Installed. Restarting now. After it restarts, run cell 2.")
import os; os.kill(os.getpid(), 9)

In [ ]:
# 2. Load the model (about 6 minutes the first time)
import torch, io, os, requests
from PIL import Image
from diffusers import Flux2KleinPipeline, PipelineQuantizationConfig

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run cell 1 again."
RAW = "https://raw.githubusercontent.com/HazeEve/movie-aniyomi/claude/gallant-lovelace-tp39bm/AuraFizz/art"
ref = Image.open(io.BytesIO(requests.get(RAW + "/generated/coffee_front_final.png").content)).convert("RGB")
ref = ref.resize((1024, 576))
batches = requests.get(RAW + "/tripo/batches.json").json()["batches"]
print(len(batches), "batches loaded")

dtype = torch.bfloat16
quant = PipelineQuantizationConfig(
    quant_backend="bitsandbytes_4bit",
    quant_kwargs={"load_in_4bit": True, "bnb_4bit_quant_type": "nf4", "bnb_4bit_compute_dtype": dtype},
    components_to_quantize=["transformer", "text_encoder"],
)
pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B",
                                          quantization_config=quant, torch_dtype=dtype)
pipe.to("cuda")
os.makedirs("/content/out", exist_ok=True)
print("Model ready. GPU memory used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

In [ ]:
# 3. Generate. Change FIRST and LAST to pick which batches (1 to 79).
FIRST, LAST = 1, 3
TRIES = 2                 # versions per batch, pick the best
WIDTH, HEIGHT = 1360, 768 # 16:9

LEAD = ("Use the attached image only as the ART STYLE reference: copy its exact outline weight, cel shading, "
        "warm lamp lighting, colours and cute chunky shapes. Do NOT copy its scene, wall, counter or objects. "
        "Draw a completely new image: ")

import time
from IPython.display import display
for n in range(FIRST, LAST + 1):
    b = batches[n - 1]
    for t in range(TRIES):
        start = time.time()
        img = pipe(image=[ref], prompt=LEAD + b["prompt"], width=WIDTH, height=HEIGHT,
                   num_inference_steps=4, guidance_scale=1.0,
                   generator=torch.Generator("cuda").manual_seed(1000 * n + t)).images[0]
        path = f"/content/out/batch{n:02d}_{t+1}.png"
        img.save(path)
        print(f"{path}  ({time.time()-start:.0f}s)  {b['title']}")
        display(img.resize((680, 384)))

In [ ]:
# 4. Download everything you made as one zip
!cd /content && zip -qr aurafizz_batches.zip out
from google.colab import files
files.download("/content/aurafizz_batches.zip")

Send me the zip. I cut every object out, name it, and colour-match it to the coffee scene.

**If a cell shows a red error:** take a screenshot of the bottom of the red text and send it to me.
Free Colab gives a few hours of GPU per day; when it stops, continue tomorrow from the next batch number (cells 1, 2, 3 again).